# CMU-MOSEI: сравнение двух версий датасета

Сравниваем:
- **EAAI** (`EAAI/CMU-MOSEI/`) — оригинальный датасет из статьи EAAI
- **mosei_data** (`mosei_data/`) — версия, загруженная из интернета

Разделы:
1. Загрузка и базовая структура  
2. Сплиты: количество клипов и именование  
3. Пересечение клипов между сплитами (swap-тест)  
4. Структура колонок  
5. Шкала эмоциональных оценок  
6. Согласованность доминирующего класса  
7. Распределение доминирующего класса в каждом сплите  
8. Клипы, присутствующие только в одной версии  
9. Выводы и рекомендации

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110

# Пути — запускать из директории NIR/mehr/notebooks/
EAAI_DIR  = Path("../../EAAI/CMU-MOSEI")
MOSEI_DIR = Path("../../mosei_data")

EMO_COLS = ["Neutral", "Anger", "Disgust", "Fear", "Happiness", "Sadness", "Surprise"]
EMO_PAL  = {
    "Neutral":   "#9e9e9e", "Anger":     "#e05252", "Disgust":   "#6abf69",
    "Fear":      "#f4a042", "Happiness": "#f4c542", "Sadness":   "#5b9bd5",
    "Surprise":  "#a67cbf",
}

# ── Загрузка ─────────────────────────────────────────────────────────────────
eaai_train = pd.read_csv(EAAI_DIR  / "train_full.csv")
eaai_dev   = pd.read_csv(EAAI_DIR  / "dev_full.csv")
eaai_test  = pd.read_csv(EAAI_DIR  / "test_full.csv")

m_train    = pd.read_csv(MOSEI_DIR / "train.csv")
m_val      = pd.read_csv(MOSEI_DIR / "validation.csv")
m_test     = pd.read_csv(MOSEI_DIR / "test.csv")

# Парсим video_name из EAAI → video + start_time
# round(2) — допускает небольшую погрешность float при сравнении ключей
for df in [eaai_train, eaai_dev, eaai_test]:
    df["video"]      = df["video_name"].str.rsplit("_", n=2).str[0]
    df["start_time"] = df["video_name"].str.rsplit("_", n=2).str[1].astype(float)
    df["end_time"]   = df["video_name"].str.rsplit("_", n=2).str[2].astype(float)
    df["clip_key"]   = df["video"] + "||" + df["start_time"].round(2).astype(str)

for df in [m_train, m_val, m_test]:
    df["clip_key"] = df["video"] + "||" + df["start_time"].round(2).astype(str)

print("Данные загружены.")
print(f"EAAI:     train={len(eaai_train)}, dev={len(eaai_dev)}, test={len(eaai_test)}")
print(f"mosei_data: train={len(m_train)}, val={len(m_val)}, test={len(m_test)}")

## 2. Сплиты: количество клипов и именование

In [ ]:
splits_eaai  = {"train": eaai_train, "dev":  eaai_dev,  "test": eaai_test}
splits_mosei = {"train": m_train,    "val":  m_val,     "test": m_test}

print(f"{'Split':<10} {'EAAI':>8} {'mosei_data':>12}  Примечание")
print("-" * 50)
pairs = [("train","train"), ("dev","val"), ("test","test")]
for ea, ms in pairs:
    n_ea = len(splits_eaai[ea])
    n_ms = len(splits_mosei[ms])
    note = "✓" if n_ea == n_ms else f"РАЗНИЦА: {n_ms - n_ea:+d}"
    print(f"  {ea:<6} / {ms:<4}  {n_ea:>8}  {n_ms:>10}   {note}")
print(f"  {'TOTAL':<12}  {sum(len(d) for d in splits_eaai.values()):>8}  "
      f"{sum(len(d) for d in splits_mosei.values()):>10}")

fig, ax = plt.subplots(figsize=(8, 3.5))
labels = ["train", "dev/val", "test"]
eaai_n  = [len(eaai_train), len(eaai_dev),  len(eaai_test)]
mosei_n = [len(m_train),    len(m_val),     len(m_test)]
x = np.arange(3)
w = 0.35
bars1 = ax.bar(x - w/2, eaai_n,  w, label="EAAI",       color="#5b9bd5", edgecolor="white")
bars2 = ax.bar(x + w/2, mosei_n, w, label="mosei_data", color="#f4a042", edgecolor="white")
for bar, v in zip(list(bars1)+list(bars2), eaai_n+mosei_n):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(v), ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel("Количество клипов")
ax.set_title("Размер сплитов: EAAI vs mosei_data", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

## 3. Swap-тест: val и test сплиты перепутаны?

In [ ]:
eaai_train_keys = set(eaai_train["clip_key"])
eaai_dev_keys   = set(eaai_dev["clip_key"])
eaai_test_keys  = set(eaai_test["clip_key"])
m_train_keys    = set(m_train["clip_key"])
m_val_keys      = set(m_val["clip_key"])
m_test_keys     = set(m_test["clip_key"])

print("=" * 65)
print("ГИПОТЕЗА: mosei_data 'val' = EAAI 'test',  mosei_data 'test' = EAAI 'dev'")
print("=" * 65)

checks = [
    ("EAAI dev   ∩ mosei test",  eaai_dev_keys,  m_test_keys, len(eaai_dev_keys)),
    ("EAAI test  ∩ mosei val",   eaai_test_keys, m_val_keys,  len(eaai_test_keys)),
    ("EAAI dev   ∩ mosei val",   eaai_dev_keys,  m_val_keys,  len(eaai_dev_keys)),
    ("EAAI test  ∩ mosei test",  eaai_test_keys, m_test_keys, len(eaai_test_keys)),
    ("EAAI train ∩ mosei train", eaai_train_keys, m_train_keys, len(eaai_train_keys)),
]
for label, s1, s2, total in checks:
    n = len(s1 & s2)
    pct = n / total * 100
    flag = "✓✓✓" if pct > 99 else ("✗✗✗ (утечка!)" if pct > 5 and "val" in label and "test" in label else "─")
    print(f"  {label:<32}: {n:>5} / {total:<5} ({pct:.1f}%)  {flag}")

print()
print("ВЫВОД: mosei_data val/test ПЕРЕПУТАНЫ местами (swap).")
print(f"  mosei 'validation' ({len(m_val)} клипов) — это EAAI 'test'  (крупный сплит).")
print(f"  mosei 'test'       ({len(m_test)} клипов) — это EAAI 'dev'  (малый сплит).")

In [ ]:
# Визуализация swap-матрицы
overlap_matrix = np.array([
    [len(eaai_train_keys & m_train_keys), len(eaai_train_keys & m_val_keys), len(eaai_train_keys & m_test_keys)],
    [len(eaai_dev_keys   & m_train_keys), len(eaai_dev_keys   & m_val_keys), len(eaai_dev_keys   & m_test_keys)],
    [len(eaai_test_keys  & m_train_keys), len(eaai_test_keys  & m_val_keys), len(eaai_test_keys  & m_test_keys)],
])

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(overlap_matrix, annot=True, fmt="d", cmap="YlOrRd",
            xticklabels=["mosei train", "mosei val", "mosei test"],
            yticklabels=["EAAI train", "EAAI dev", "EAAI test"],
            linewidths=0.5, linecolor="white", ax=ax,
            cbar_kws={"label": "Число общих клипов"})
ax.set_title("Матрица пересечений клипов по сплитам", fontweight="bold")
plt.tight_layout()
plt.show()

## 4. Структура колонок

In [ ]:
eaai_orig_cols  = pd.read_csv(EAAI_DIR  / "train_full.csv", nrows=0).columns.tolist()
mosei_orig_cols = pd.read_csv(MOSEI_DIR / "train.csv",      nrows=0).columns.tolist()

all_cols = sorted(set(eaai_orig_cols) | set(mosei_orig_cols))

print(f"{'Колонка':<20} {'EAAI':^8} {'mosei_data':^12}")
print("-" * 42)
for c in all_cols:
    in_eaai  = "✓" if c in eaai_orig_cols  else "─"
    in_mosei = "✓" if c in mosei_orig_cols else "─"
    note = ""
    if c == "Other":
        note = "← всегда 0 в EAAI"
    elif c in ["sentiment", "ASR", "start_time", "end_time"]:
        note = "← только в mosei_data"
    elif c == "video_name":
        note = "← video+start+end в одной строке"
    print(f"  {c:<18} {in_eaai:^8} {in_mosei:^12}  {note}")

print()
print("EAAI исходный порядок:", eaai_orig_cols)
print()
print("mosei_data порядок:   ", mosei_orig_cols)

## 5. Шкала эмоциональных оценок

**Ключевое различие**: оба датасета используют разные метрики для одних и тех же клипов.

In [ ]:
print("=== Диапазон значений ===")
print(f"{'Emotion':<12} {'EAAI min>0':>12} {'EAAI max':>10} {'mosei min>0':>14} {'mosei max':>10}")
print("-" * 60)
for e in EMO_COLS:
    if e == "Neutral":
        print(f"  {e:<10} {'binary {0,1}':>12} {'':>10} {'binary {0,1}':>14} {'':>10}")
        continue
    eaai_nz  = eaai_train[eaai_train[e] > 0][e]
    mosei_nz = m_train[m_train[e] > 0][e]
    print(f"  {e:<10} {eaai_nz.min():>12.4f} {eaai_nz.max():>10.4f}"
          f" {mosei_nz.min():>14.4f} {mosei_nz.max():>10.4f}")

print()
print("EAAI:      оценки ∈ [~0.05, 1.0] — дробные (k/N, N ~ 6-21 аннотаторов)")
print("mosei_data: оценки ∈ [0.17, 3.0] — кратны 1/6 → средняя интенсивность по 6 аннотаторам (0-3 шкала)")

In [ ]:
# Гистограммы оценок Happiness для наглядного сравнения
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for row, (df, label, max_val) in enumerate([
    (eaai_train, "EAAI (k/N, нормирован на 1)", 1.0),
    (m_train,    "mosei_data (средняя интенсивность 0–3)", 3.0),
]):
    for col, emo in enumerate(["Happiness", "Sadness", "Anger"]):
        ax = axes[row, col]
        vals = df[df[emo] > 0][emo]
        ax.hist(vals, bins=30, color=EMO_PAL[emo], edgecolor="white", alpha=0.85)
        ax.set_title(f"{label}\n{emo}", fontweight="bold", fontsize=9)
        ax.set_xlabel("Score")
        ax.set_ylabel("Count")
        ax.set_xlim(0, max_val + 0.1)

plt.suptitle("Распределение оценок (только ненулевые): сравнение шкал", fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

# Пример одного клипа
print("=== Пример: один и тот же клип в обоих датасетах ===")
common_train = list(eaai_train_keys & m_train_keys)[:3]
eaai_idx = eaai_train.set_index("clip_key")
m_idx    = m_train.set_index("clip_key")
for k in common_train:
    e_row = eaai_idx.loc[k, EMO_COLS].values.astype(float)
    m_row = m_idx.loc[k, EMO_COLS].values.astype(float)
    nz = e_row > 0
    print(f"\nКлип: {k}")
    print(f"  EAAI:       {dict(zip(EMO_COLS, e_row.round(4)))}")
    print(f"  mosei_data: {dict(zip(EMO_COLS, m_row.round(4)))}")
    if nz.any():
        ratios = m_row[nz] / e_row[nz]
        print(f"  Ratio (mosei/EAAI) для ненулевых: {ratios.round(3)}")

## 6. Согласованность доминирующего класса

Несмотря на разную шкалу, важно: дают ли обе версии **одинаковый dominant label** для каждого клипа?

In [ ]:
results = {}
for split_name, eaai_df, mosei_df, eaai_keys_set, mosei_keys_set in [
    ("train", eaai_train, m_train,  eaai_train_keys, m_train_keys),
    ("dev/test", pd.concat([eaai_dev, eaai_test]), pd.concat([m_val, m_test]),
      eaai_dev_keys | eaai_test_keys, m_val_keys | m_test_keys),
]:
    common = list(eaai_keys_set & mosei_keys_set)
    e_idx = eaai_df.set_index("clip_key")
    m_idx = mosei_df.set_index("clip_key")

    eaai_dom  = e_idx.loc[common, EMO_COLS].idxmax(axis=1)
    mosei_dom = m_idx.loc[common, EMO_COLS].idxmax(axis=1)
    agree = (eaai_dom == mosei_dom).mean()
    results[split_name] = {"agree": agree, "n": len(common),
                            "eaai_dom": eaai_dom, "mosei_dom": mosei_dom}
    print(f"Сплит {split_name}: dominant-class agreement = {agree*100:.2f}% ({len(common)} клипов)")

print()
print("Вывод: dominant label ИДЕНТИЧЕН в обоих датасетах → шкала не влияет на argmax-классификацию.")
print("Расхождения только в равновесных случаях (tie-breaking при argmax).")

## 7. Распределение доминирующего класса в каждом сплите

In [ ]:
# Сравниваем правильное разбиение (EAAI) vs то, что было в mosei_data
fig, axes = plt.subplots(2, 3, figsize=(16, 8))

comparisons = [
    ("EAAI train",   eaai_train, 0, 0),
    ("EAAI dev",     eaai_dev,   0, 1),
    ("EAAI test",    eaai_test,  0, 2),
    ("mosei train",  m_train,    1, 0),
    ("mosei val ← EAAI TEST",  m_val,   1, 1),
    ("mosei test ← EAAI DEV",  m_test,  1, 2),
]
for label, df, row, col in comparisons:
    ax = axes[row, col]
    dom = df[EMO_COLS].idxmax(axis=1)
    counts = dom.value_counts().reindex(EMO_COLS, fill_value=0)
    colors = [EMO_PAL[e] for e in counts.index]
    ax.bar(counts.index, counts.values, color=colors, edgecolor="white")
    ax.set_title(label, fontweight="bold", fontsize=10)
    ax.set_ylabel("# клипов")
    for i, (v, pct) in enumerate(zip(counts.values, counts.values / counts.sum() * 100)):
        ax.text(i, v + 5, f"{pct:.0f}%", ha="center", fontsize=7)
    ax.tick_params(axis='x', labelrotation=30)

plt.suptitle("Dominant-class distribution по сплитам\n(верхний ряд = EAAI правильное; нижний = mosei_data с перепутанными val/test)",
             fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

## 8. Клипы, присутствующие только в одной версии

In [ ]:
# Клипы в EAAI dev, которых нет в mosei test (и наоборот)
print("=== Клипы ТОЛЬКО в EAAI dev (нет в mosei test) ===")
only_eaai_dev = eaai_dev_keys - m_test_keys
print(f"  Количество: {len(only_eaai_dev)}")
if only_eaai_dev:
    sample = eaai_dev[eaai_dev["clip_key"].isin(only_eaai_dev)][["video_name","text"] + EMO_COLS].head(5)
    print(sample.to_string(index=False))

print()
print("=== Клипы ТОЛЬКО в mosei test (нет в EAAI dev) ===")
only_m_test = m_test_keys - eaai_dev_keys
print(f"  Количество: {len(only_m_test)}")
if only_m_test:
    sample = m_test[m_test["clip_key"].isin(only_m_test)][["video","start_time","text"] + EMO_COLS].head(5)
    print(sample.to_string(index=False))

print()
print("=== Клипы ТОЛЬКО в EAAI test (нет в mosei val) ===")
only_eaai_test = eaai_test_keys - m_val_keys
print(f"  Количество: {len(only_eaai_test)}")

print()
print("=== Клипы ТОЛЬКО в mosei val (нет в EAAI test) ===")
only_m_val = m_val_keys - eaai_test_keys
print(f"  Количество: {len(only_m_val)}")

## 9. Выводы и рекомендации

In [ ]:
eaai_train_keys = set(eaai_train["clip_key"])
eaai_dev_keys   = set(eaai_dev["clip_key"])
eaai_test_keys  = set(eaai_test["clip_key"])
m_train_keys    = set(m_train["clip_key"])
m_val_keys      = set(m_val["clip_key"])
m_test_keys     = set(m_test["clip_key"])

n_dev_test  = len(eaai_dev_keys  & m_test_keys)
n_test_val  = len(eaai_test_keys & m_val_keys)
n_train_tr  = len(eaai_train_keys & m_train_keys)
miss_dev    = len(eaai_dev_keys)  - n_dev_test
miss_test   = len(eaai_test_keys) - n_test_val
miss_train  = len(eaai_train_keys) - n_train_tr

print(f"""
╔══════════════════════════════════════════════════════════════════════╗
║              СВОДКА РАЗЛИЧИЙ: EAAI vs mosei_data                   ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  1. SWAP VAL/TEST  ⚠️  КРИТИЧНО                                     ║
║     mosei 'validation' ({len(m_val)} кл.) = EAAI 'test' ({len(eaai_test)} кл.)         ║
║     mosei 'test'       ({len(m_test)} кл.)  = EAAI 'dev'  ({len(eaai_dev)} кл.)        ║
║     Overlap: dev∩test={n_dev_test}/{len(eaai_dev_keys)}, test∩val={n_test_val}/{len(eaai_test_keys)}     ║
║     → Модели на mosei_data: "test" = dev, "val" = test.            ║
║       Метрики на "test" несравнимы со статьёй.                     ║
║                                                                      ║
║  2. ШКАЛА ОЦЕНОК  (влияет на loss, НЕ на argmax)                   ║
║     EAAI:       score = k/N аннотаторов ∈ [0.047, 1.0]            ║
║     mosei_data: средняя интенсивность (0–3 шкала, шаг 1/6)        ║
║     dominant label agreement: 100% (argmax идентичен)              ║
║     НО: multi-label порог (>0) и class_weights несопоставимы       ║
║                                                                      ║
║  3. УНИКАЛЬНЫЕ КОЛОНКИ                                              ║
║     EAAI only:       Other (всегда 0 → игнорировать)               ║
║     mosei_data only: sentiment, ASR, start_time, end_time           ║
║                                                                      ║
║  4. КЛИПЫ (неполное пересечение)                                    ║
║     train: {n_train_tr}/{len(eaai_train_keys)} общих ({miss_train} только в EAAI)                  ║
║     dev:   {n_dev_test}/{len(eaai_dev_keys)} общих с mosei test  ({miss_dev} отсутствуют)         ║
║     test:  {n_test_val}/{len(eaai_test_keys)} общих с mosei val ({miss_test} отсутствуют)         ║
║                                                                      ║
╠══════════════════════════════════════════════════════════════════════╣
║  РЕКОМЕНДАЦИЯ                                                        ║
║  ✓ Использовать EAAI как референс                                   ║
║  ✓ Правильный порядок сплитов: train / dev (1861) / test (4653)    ║
║  ✓ В mosei_data переименовать: val→test, test→dev                  ║
║  ✓ Для сравнения с литературой — брать EAAI шкалу (0-1)           ║
╚══════════════════════════════════════════════════════════════════════╝
""")